Running two unbiased simulation for site A and B of alanine dipeptide

In [3]:
import os
import subprocess
from pathlib import Path
import numpy as np
import math
import plumed
import matplotlib.pyplot as plt

In [4]:
# set working directory for all future cells
os.chdir("/home/dani/wslcoding/MCFM/ML-CV")
# and check with bash command pwd - note the exclamation mark at the beginning
!pwd
# or in python
os.getcwd()

/home/dani/wslcoding/MCFM/ML-CV


'/home/dani/wslcoding/MCFM/ML-CV'

In [6]:
# delete outputs of simulations from `folder``
def clean(folder='./'):
    subprocess.run("rm bck.* COLVAR KERNELS alanine.*", cwd=folder, shell=True)

# execute bash command in the given folder
def execute(command, folder, background=False):
    cmd = subprocess.run(command, cwd=folder, shell=True, capture_output = True, text=True, close_fds=background)
    if cmd.returncode == 0:
        print(f'Completed: {command}')
    else:
        print(cmd.stderr)

#GMX_CMD = '. /work/sourceme.sh && gmx_mpi'
GMX_CMD = 'gmx_mpi'

In [ ]:
# CREATE FOLDER AND COPY INPUTS
folder = 'sample/0_unbiased-sA/'
Path(folder).mkdir(parents=True, exist_ok=True)
execute(f"cp ../../input/md_inputs/input.ala2.pdb ../../input/md_inputs/input.tpr .", folder=folder)

# WRITE PLUMED INPUT FILE
with open(folder+"plumed.dat","w") as f:
    print("""
# vim:ft=plumed

# Compute torsion angles, as well as energy
MOLINFO STRUCTURE=input.ala2.pdb
phi: TORSION ATOMS=@phi-2
psi: TORSION ATOMS=@psi-2
theta: TORSION ATOMS=6,5,7,9
xi: TORSION ATOMS=16,15,17,19
ene: ENERGY

# Compute descriptors
INCLUDE FILE=../../input/plumed-distances.dat

# Print 
PRINT FMT=%g STRIDE=100 FILE=COLVAR ARG=*

ENDPLUMED
""",file=f)

Completed: cp ../../input/md_inputs/input.ala2.pdb ../../input/md_inputs/input.tpr .


In [15]:
## RUN GROMACS
#num_steps=500000
num_steps=10_000

#clean(folder) # note: this deletes all previous results in folder!
execute(f"{GMX_CMD} mdrun -s input.tpr -deffnm alanine -plumed plumed.dat -ntomp 1 -nsteps {num_steps} -v > alanine.out", folder=folder)

Completed: gmx_mpi mdrun -s input.tpr -deffnm alanine -plumed plumed.dat -ntomp 1 -nsteps 10000 -v > alanine.out
